# 集合覆盖 scp41（最小成本）

**问题**：实例来自 OR-Library 的 scp41：m=200 个行元素，n=1000 个列集合。列 j 的成本为 c_j，覆盖的行集合为 S_j（a_ij=1 表示列 j 覆盖行 i）。目标是选择一组列，使每一行至少被一个选中列覆盖，同时总成本最小。

**数学模型**

$$\min \sum_{j=1}^{n} c_j x_j$$

$$\text{s.t.}\quad \sum_{j: i \in S_j} x_j \ge 1,\quad i=1,\dots,m$$

$$x_j\in\{0,1\},\quad j=1,\dots,n$$

数据文件：\`/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt\`。文献最优值 429（本套件用直接 MIP 自证）。

## 方法：直接混合整数规划（HIGHS）

**原理要点**

1. 直接把 0-1 集合覆盖模型交给 MathOpt + HIGHS 求解。
2. 目标是最小化总成本，200 个覆盖约束全部显式加入。
3. HIGHS 在分支定界中同时给出 primal bound 与 dual bound；两者相等即为证明最优。
4. 停机条件：\`time_limit=120s\`（本实例远小于该上限）、\`enable_output=False\`。
5. 本实例规模很小（1000 个 0-1 变量、200 个约束），直接 MIP 是最可靠的基准。

**实现要点**

- 变量 \`x[j]\`：\`add_variable(lb=0, ub=1, is_integer=True)\`。
- 约束：对每一行 \`add_linear_constraint(sum(x[j] for j in row) >= 1)\`。
- 结果读取：\`res.termination.reason\`、\`res.objective_value()\`、\`res.best_objective_bound()\`。
- 求解后用列-行邻接表 \`colrows\` 独立核验覆盖数与目标值。

In [1]:
import platform, time, datetime, math, ortools
from ortools.math_opt.python import mathopt

print("python", platform.python_version(), "| ortools", ortools.__version__)

DATA = "/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt"
toks = open(DATA).read().split()
m, n = map(int, toks[:2])
costs = list(map(int, toks[2:2+n]))
idx = 2 + n
rows = []
for _ in range(m):
    k = int(toks[idx]); idx += 1
    rows.append([int(t)-1 for t in toks[idx:idx+k]]); idx += k
assert idx == len(toks)
colrows = [[] for _ in range(n)]
for i, row in enumerate(rows):
    for j in row:
        colrows[j].append(i)
print("m,n =", m, n, "| rows parsed =", len(rows), "| tokens consumed =", idx)


python 3.10.20 | ortools 9.15.6755
m,n = 200 1000 | rows parsed = 200 | tokens consumed = 5211


In [2]:
t0 = time.perf_counter()
model = mathopt.Model(name="scp41_direct")
x = [model.add_variable(lb=0.0, ub=1.0, is_integer=True, name=f"x{j}") for j in range(n)]
model.minimize_linear_objective(sum(costs[j]*x[j] for j in range(n)))
for i, row in enumerate(rows):
    model.add_linear_constraint(sum(x[j] for j in row) >= 1.0, name=f"cov{i}")
params = mathopt.SolveParameters(time_limit=datetime.timedelta(seconds=120), enable_output=False)
res = mathopt.solve(model, mathopt.SolverType.HIGHS, params=params)
wall = time.perf_counter() - t0
vals = res.variable_values(x)
sel = [j for j in range(n) if vals[j] > 0.5]
covered = [False]*m
for j in sel:
    for i in colrows[j]:
        covered[i] = True
print("termination:", res.termination.reason)
print("objective:", res.objective_value())
print("best_bound:", res.best_objective_bound())
print("solve_time:", res.solve_time(), "| wall:", round(wall, 3))
print("num_selected:", len(sel), "| obj_check:", sum(costs[j] for j in sel), "| covered_rows:", sum(covered), "/", m)
print("selected_columns:", sorted(sel))


termination: TerminationReason.OPTIMAL
objective: 429.0
best_bound: 429.0
solve_time: 0:00:00.070216 | wall: 0.25
num_selected: 66 | obj_check: 429 | covered_rows: 200 / 200
selected_columns: [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 24, 25, 27, 28, 42, 43, 45, 46, 47, 48, 49, 51, 53, 57, 58, 61, 62, 65, 68, 69, 70, 74, 76, 77, 80, 84, 85, 88, 90, 93, 102, 106, 115, 119, 120, 121, 123, 128, 137, 142, 143, 145, 152, 193, 274, 432]


## 运行结果与结论

上方输出显示 HIGHS 以 \`TerminationReason.OPTIMAL\` 结束，目标值 **429.0**，best bound 同为 429.0，证明最优。选择 66 列，覆盖全部 200 行。

**基准最优值来源**：本 notebook 直接 MIP 自证最优值 429.0，与 OR-Library 文献值 429 一致。

## 结论

直接 MIP 对 scp41 规模足够快且能证明最优，是其余四个分解方法的基准。